In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
import torch
import torch.nn as nn   
import torch.optim as optim
import torch.nn.functional as F

from torch.optim import SGD

from torch.utils.data import DataLoader, TensorDataset, random_split

In [3]:
import h5py
from tqdm import tqdm

In [4]:
import os
import gc

In [5]:
comet_key = os.environ.get("comet_ml_key")

In [6]:
import comet_ml
# comet_ml.login(comet_key)

In [7]:
exp = comet_ml.start(api_key = comet_key, project_name = "Dissertation ML")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/alcatraz312/dissertation-ml/3b05bbc5cd414115ad3bb81457edc3b1



#### Data prepare pipeline

In [8]:
data_path = "/home/arbiter/projects/Survey-invariant-generalization/data/usable_data"

In [9]:
sdss_data = h5py.File(f"{data_path}/sdss_resampled.h5")
len(sdss_data.keys())

21960

In [10]:
sdss_meta_data = pd.read_csv(f"{data_path}/sdss_meta.csv")
sdss_meta_data.shape

(21998, 13)

In [11]:
def get_flux_labels(data, meta_data):
    
    ''' 
    Preparing the features and labels for the experiments \n
    flux and their corresponding spectral MK classes and atmospheric parameters 
    '''

    flux_list     = []
    teff_list     = []
    logg_list     = []
    feh_list      = []
    cls_list      = []
    failed        = 0

    # Map spectral class letter to integer index
    class_map = {"O": 0, "B": 1, "A": 2, "F": 3, "G": 4, "K": 5, "M": 6}

    for star_id in tqdm(data):
        star_id_info = star_id.split("-")   # split into list containing the plate, fiber id and mjd for the star

        try:
            # Flux
            flux_array = data[star_id]["flux"][:]

            # Metadata lookup
            star_meta = meta_data.loc[
                (meta_data["plate"]   == int(star_id_info[0])) &
                (meta_data["fiberid"] == int(star_id_info[1])) &
                (meta_data["mjd"]     == int(star_id_info[2]))
            ]

            # Skip if no match found
            if len(star_meta) == 0:
                failed += 1
                continue

            teff = star_meta["elodieTeff"].iloc[0]
            logg = star_meta["elodieLogG"].iloc[0]
            feh  = star_meta["elodieFeH"].iloc[0]
            spectral_class = star_meta["subclass"].iloc[0][0]

            # Skip if any label is null
            if pd.isna(teff) or pd.isna(logg) or pd.isna(feh):
                failed += 1
                continue

            # Skip if class not in map
            if spectral_class not in class_map:
                failed += 1
                continue

            flux_list.append(flux_array)
            teff_list.append(teff)
            logg_list.append(logg)
            feh_list.append(feh)
            cls_list.append(class_map[spectral_class])

        except Exception as e:
            print(f"Failed for {star_id}: {e}")
            failed += 1
            continue

    print(f"Collected: {len(flux_list)} | Failed/skipped: {failed}")

    # Stack into arrays
    flux_array  = np.array(flux_list)                          # (n_stars, 3800)
    y_reg       = np.column_stack([teff_list, logg_list, feh_list])  # (n_stars, 3)
    y_cls       = np.array(cls_list)                           # (n_stars,)

    return flux_array, y_reg, y_cls

In [12]:
# flux, y_reg, y_cls = get_flux_labels(sdss_data, sdss_meta_data)

In [13]:
def normalise_labels(y_reg):
    '''
    Standardise each parameter to zero mean unit variance.
    Returns normalised array + stats for denormalisation later.
    '''
    mu    = y_reg.mean(axis=0)    # shape -> (3,)
    sigma = y_reg.std(axis=0)     # shape -> (3,)
    return (y_reg - mu) / sigma, mu, sigma


def denormalise_labels(y_norm, mu, sigma):
    return y_norm * sigma + mu

In [14]:
def prepare_dataloader(flux, y_reg, y_cls, 
                       val_test_split=0.4, batch_size=256, seed=42):

    ''' 
    Preparing the tensor sets for flux and labels
    '''

    flux_tensor  = torch.FloatTensor(flux)    # rank 1 tensors
    y_reg_tensor = torch.FloatTensor(y_reg)
    y_cls_tensor = torch.LongTensor(y_cls)     # tensor with int values

    # Shuffle before splitting
    n    = len(flux_tensor)
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    flux_tensor  = flux_tensor[perm]
    y_reg_tensor = y_reg_tensor[perm]
    y_cls_tensor = y_cls_tensor[perm]

    dataset = TensorDataset(flux_tensor, y_reg_tensor, y_cls_tensor)

    # Compute integer sizes
    n_val_test = int(n * val_test_split)
    n_train    = n - n_val_test
    n_val      = n_val_test // 2
    n_test     = n_val_test - n_val    # handles odd numbers cleanly

    # splitting the datasets into train, val and test datasets

    train_set, val_set, test_set = random_split(
        dataset, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(seed)
    )

    train_loader = DataLoader(train_set, batch_size=batch_size,
                              shuffle=True,  drop_last=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size,
                              shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_set,  batch_size=batch_size,
                              shuffle=False, drop_last=False)

    print(f"Train: {n_train} | Val: {n_val} | Test: {n_test}")
    return train_loader, val_loader, test_loader

## Multi task learning architecture

In [15]:
class MTLArchitecture(nn.Module):
    def __init__(self, input_dim, latent_dim, num_classes):
        '''
        input_dim  : number of wavelength pixels -> int (3800)
        latent_dim : dimension of latent space   -> int
        num_classes: number of MK classes        -> int
        '''
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
        )

        # Latent distribution
        self.mu_layer     = nn.Linear(128, latent_dim)
        self.logvar_layer = nn.Linear(128, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_dim)
        )

        # Learned global log variance for reconstruction
        # scalar — shared across all pixels
        # avoids conflating encoder uncertainty with reconstruction uncertainty
        self.log_sigma_recon = nn.Parameter(torch.zeros(1))

        # Regression head (heteroscedastic)
        # predicts mean + uncertainty for each atmospheric parameter
        self.regression_head = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.reg_mu     = nn.Linear(32, 3)   # Teff, log g, [Fe/H]
        self.reg_logvar = nn.Linear(32, 3)   # per-parameter log variance

        # Classification head 
        self.classification_head = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

        # homoscedastic uncertainties for each loss
        self.logvar_recon = nn.Parameter(torch.zeros(1))    
        self.logvar_reg = nn.Parameter(torch.zeros(1))
        self.logvar_cls = nn.Parameter(torch.zeros(1))

    # Reparameterization
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # forward pass
    def forward(self, x):
        # Encode
        h      = self.encoder(x)
        mu     = self.mu_layer(h)
        logvar = self.logvar_layer(h)

        # Sample latent vector
        z = self.reparameterize(mu, logvar)

        # Decode
        x_hat = self.decoder(z)

        # Regression head
        z_reg       = self.regression_head(z)
        mu_reg      = self.reg_mu(z_reg)       # (batch, 3)
        logvar_reg  = self.reg_logvar(z_reg)   # (batch, 3)

        # Classification head
        cls_logits = self.classification_head(z)   # (batch, num_classes)

        return {
            "x_hat"      : x_hat,        # reconstructed spectrum
            "mu"         : mu,           # latent mean
            "logvar"     : logvar,       # latent log variance
            "mu_reg"     : mu_reg,       # predicted atmospheric params
            "logvar_reg" : logvar_reg,   # predicted param uncertainties
            "cls_logits" : cls_logits    # raw class logits
        }

    # Loss components

    def reconstruction_loss(self, x, x_hat):
        '''
        Gaussian NLL with learned global variance
        log_sigma_recon is a learned scalar parameter
        mean over pixels, mean over batch → scalar
        '''
        log_sigma = torch.clamp(self.log_sigma_recon, -10, 5)
        var       = torch.exp(2 * log_sigma)

        nll = 0.5 * torch.mean(
            2 * log_sigma + (x - x_hat)**2 / var,
            dim=1                                    # mean over pixels
        )
        return nll.mean()                            # mean over batch → scalar

    def kl_divergence(self, mu, logvar):
        '''
        KL( q(z|x) || N(0,I) )
        sum over latent dims, mean over batch → scalar
        '''
        logvar = torch.clamp(logvar, -10, 5)
        kl = 0.5 * torch.sum(
            mu**2 + torch.exp(logvar) - 1 - logvar,
            dim=1                                    # sum over latent dims
        )
        return kl.mean()                             # mean over batch → scalar

    def regression_loss(self, mu_reg, logvar_reg, y):
        '''
        Heteroscedastic Gaussian NLL for atmospheric parameters
        y     : ground truth params (batch, 3)
        mean over params, mean over batch → scalar
        '''
        logvar_reg = torch.clamp(logvar_reg, -10, 5)
        nll = 0.5 * torch.mean(
            logvar_reg + (y - mu_reg)**2 / torch.exp(logvar_reg),
            dim=1                                    # mean over 3 params
        )
        return nll.mean()                            # mean over batch → scalar

    def classification_loss(self, cls_logits, labels):
        '''
        Standard cross entropy
        labels : integer class indices (batch,)
        '''
        return F.cross_entropy(cls_logits, labels)
    
    def aggregate_loss(self,x, x_hat, mu, logvar, mu_reg, logvar_reg, y, cls_logits, labels, beta = 1.0):
        '''
        Aggregated loss \n
        negative Evidence lower bound + regression loss + classification loss
        '''

        recon_loss = self.reconstruction_loss(x, x_hat)
        kl = self.kl_divergence(mu, logvar)
        reg_loss = self.regression_loss(mu_reg, logvar_reg, y)
        cls_loss = self.classification_loss(cls_logits, labels)

        # uncertainty weights for each loss function
        w_recon = 0.5 * torch.exp(-self.logvar_recon)
        w_cls = 0.5 * torch.exp(-self.logvar_cls)
        w_reg = 0.5 * torch.exp(-self.logvar_reg)

        loss = (w_recon * recon_loss + self.logvar_recon + beta * kl) + (w_reg * reg_loss + 0.5 * self.logvar_reg) + (w_cls * cls_loss + 0.5 * self.logvar_cls)

        components = {
            "loss" : loss.item(),
            "recon" : recon_loss.item(),
            "kl" : kl.item(),
            "reg" : reg_loss.item(),
            "cls" : cls_loss.item(),

            "sigma_recon" : torch.exp(0.5 * self.logvar_recon).item(),
            "sigma_reg"  : torch.exp(0.5 * self.logvar_reg).item(),
            "sigma_cls"  : torch.exp(0.5 * self.logvar_cls).item()
        }

        return loss, components

In [16]:
model = MTLArchitecture(input_dim = 3800, latent_dim= 64, num_classes= 7)
model

MTLArchitecture(
  (encoder): Sequential(
    (0): Linear(in_features=3800, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=128, bias=True)
    (5): ReLU()
  )
  (mu_layer): Linear(in_features=128, out_features=64, bias=True)
  (logvar_layer): Linear(in_features=128, out_features=64, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1024, bias=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=3800, bias=True)
  )
  (regression_head): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (reg_mu): Linear(in_features=32, out_features=3, bias=True)
  (

Defining single training step

In [17]:
def training_step(model, batch, optimizer, device, beta):

    ''' 
    Training of a batch exactly once \n
    Parameters : \n model : Torch model\n
    batch : single data batch \n
    optimizer : torch optimizer for loss propagation \n
    device : cuda or cpu for matrix multiplication \n
    beta : Kl divergence weight
    '''

    model.train()    # training mode

    flux, y_reg, y_cls = [b.to(device) for b in batch]   # load the batch into cuda

    optimizer.zero_grad()    # zero the gradient
    out = model(flux)     # feed forward the flux data

    # aggregating losses
    loss, components = model.aggregate_loss(
        x = flux,
        x_hat = out["x_hat"],
        mu = out["mu"],
        logvar = out["logvar"],
        mu_reg = out["mu_reg"],
        logvar_reg = out["logvar_reg"],
        y = y_reg,
        cls_logits = out["cls_logits"],
        labels = y_cls,
        beta = beta        
    )

    loss.backward()    # back propagation

    # Gradient clipping — prevents exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)    # prevents gradient overshoot 

    optimizer.step()    # take a discent step
    return components
    

Defining single validation step

In [18]:
@torch.no_grad()
def validation_step(model, batch, device, beta):

    ''' 
    Validation of a batch exactly once \n
    Parameters : \n model : Torch model\n
    batch : single data batch \n
    device : cuda or cpu for matrix multiplication \n
    beta : Kl divergence weight
    '''
    
    model.eval()   # validation mode

    flux, y_reg, y_cls = [b.to(device) for b in batch]
    out = model(flux) 

    loss, components = model.aggregate_loss(
        x = flux,
        x_hat = out["x_hat"],
        mu = out["mu"],
        logvar = out["logvar"],
        mu_reg = out["mu_reg"],
        logvar_reg = out["logvar_reg"],
        y = y_reg,
        cls_logits = out["cls_logits"],
        labels = y_cls,
        beta = beta        
    )

    # classification accuracy
    preds = out["cls_logits"].argmax(dim = 1)    # predicted class indices 

    classification_accuracy_boolean = (preds == y_cls)    # element wise comparison --> boolean tensor
    classification_accuracy_float = classification_accuracy_boolean.float()        # float tensor
    classification_accuracy = classification_accuracy_float.mean().item()        # mean accuracy

    # regression MAE per parameter

    mae = (out["mu_reg"] - y_reg).abs().mean(dim = 0).cpu().numpy()   # shape --> (3,)

    return components, classification_accuracy, mae


In [19]:
def train(model, train_loader, val_loader, n_epochs = 10, lr = 0.001, beta = 1.0):

    ''' 
    Training loop \n
    Parameters : \n
    model : torch model architecture \n
    train_loader = train set loader \n
    val_loader = validation set loader \n
    n_epochs = number of epochs, default set to 10 \n
    lr : learning rate of the model, defaul set to 0.001 \n
    beta : kl divergence weight
    '''

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    # transfer to cuda cores

    model = model.to(device)

    # log hyperparameters into comet ML

    exp.log_parameters({
        "n_epochs"   : n_epochs,
        "lr"         : lr,
        "beta"       : beta,
        "latent_dim" : model.mu_layer.out_features,
        "input_dim"  : model.encoder[0].in_features,
        "batch_size" : train_loader.batch_size,
    })

    # dictionary to track losses and accuracy metrics

    track_losses = {
        "train_loss" : [],
        "train_recon" : [],
        "train_kl" : [],
        "train_reg" : [],
        "train_cls" : [],
        "val_acc" : [],
        "val_mae" : [],
        "val_loss" : [],
        "val_recon" : [],
        "val_kl" : [],
        "val_reg" : [],
        "val_cls" : [],
    }

    # best_val_loss = float("inf")
    # best_epoch = 0
    # patience_count = 0

    print(f"Training on {device}")

    # training and validation loop

    for epoch in range(1, n_epochs + 1):

        # traning:

        train_components = {"loss" : 0, "recon" : 0, "kl" : 0, "reg" : 0, "cls" : 0}    # initialize loss dictionary for training for each epoch
        n_train_batches = 0           # number of batches in the training 

        for batch in train_loader:
            components = training_step(model, batch, optimizer, device, beta)     # training the model
            for k in train_components:
                train_components[k] += components[k]       # initializing the values into the loss dictionary

            n_train_batches += 1

        train_avg = {k : v/n_train_batches for k,v in train_components.items()}     # average the losses over all batches for one epoch 

        # validation

        val_components = {"loss" : 0, "recon" : 0, "kl" : 0, "reg" : 0, "cls" : 0}

        # validation error metrics
        n_val_batches = 0
        total_acc = 0.0
        total_mae = np.zeros(3)
 
        for batch in val_loader:
            components, accuracy, mae = validation_step(model, batch, device, beta)

            for k in val_components:
                val_components[k] += components[k]

            n_val_batches += 1
            total_acc += accuracy    # total accuracy of all batches for one epoch
            total_mae += mae      # total mean absolute error of all batches for one epoch

        # averaging loss and error metrics over number of batches
        val_avg = {k : v/n_val_batches for k,v in val_components.items()}
        acc_avg = total_acc/n_val_batches
        mae_avg = total_mae/n_val_batches

        # update loss tracks

        track_losses["train_loss"].append(train_avg["loss"])
        track_losses["train_recon"].append(train_avg["recon"])
        track_losses["train_kl"].append(train_avg["kl"])
        track_losses["train_cls"].append(train_avg["cls"])
        track_losses["train_reg"].append(train_avg["reg"])

        track_losses["val_loss"].append(val_avg["loss"])
        track_losses["val_recon"].append(val_avg["recon"])
        track_losses["val_kl"].append(val_avg["kl"])
        track_losses["val_cls"].append(val_avg["cls"])
        track_losses["val_reg"].append(val_avg["reg"])
        track_losses["val_acc"].append(acc_avg)
        track_losses["val_mae"].append(mae_avg)
        
        # log the experiment loss metrics and parameters to comet ml 

        exp.log_metrics({
            # Train losses
            "train/loss"  : train_avg["loss"],
            "train/recon" : train_avg["recon"],
            "train/kl"    : train_avg["kl"],
            "train/reg"   : train_avg["reg"],
            "train/cls"   : train_avg["cls"],
            # Val losses
            "val/loss"    : val_avg["loss"],
            "val/recon"   : val_avg["recon"],
            "val/kl"      : val_avg["kl"],
            "val/reg"     : val_avg["reg"],
            "val/cls"     : val_avg["cls"],
            # Val metrics
            "val/accuracy"     : acc_avg,
            "val/mae_teff"     : mae_avg[0],
            "val/mae_logg"     : mae_avg[1],
            "val/mae_feh"      : mae_avg[2],
            # Learned task uncertainties
            "sigma/recon" : torch.exp(0.5 * model.logvar_recon).item(),
            "sigma/reg"   : torch.exp(0.5 * model.logvar_reg).item(),
            "sigma/cls"   : torch.exp(0.5 * model.logvar_cls).item(),
        }, epoch=epoch)

    exp.end()

    return exp

#### Experiment

In [20]:
flux, y_reg, y_cls = get_flux_labels(sdss_data, sdss_meta_data)

100%|██████████| 21960/21960 [00:31<00:00, 706.45it/s]


Collected: 21960 | Failed/skipped: 0


In [21]:
gc.collect()

423

In [22]:
# y_reg_norm, reg_mu, reg_sigma = normalise_labels(y_reg)

Masking bad median stars (outliers and noisy pixel stars):

In [23]:
def good_stars_data(flux, y_reg, y_cls):
    per_star_median = np.median(flux , axis = 1)
    good_mask = per_star_median < 5.0

    flux_clean = flux[good_mask]
    y_reg_clean = y_reg[good_mask]
    y_cls_clean = y_cls[good_mask]

    flux_clean = np.clip(flux_clean, -3.0, 10)

    return flux_clean, y_reg_clean, y_cls_clean

In [24]:
flux_clean, y_reg, y_cls = good_stars_data(flux, y_reg, y_cls)

In [25]:
y_reg_norm, reg_mu, reg_sigma = normalise_labels(y_reg)

saving the mean and standard deviations to later retrieve the original atmospheric parameters

In [26]:
np.save("reg_mu.npy", reg_mu)     # (3,)  mean of atmospheric parameters 
np.save("reg_sigma.npy", reg_sigma)     # (3,)  standard deviation of atmospheric parameters 

In [27]:
train_loader, val_loader, test_loader = prepare_dataloader(
    flux= flux_clean,
    y_reg= y_reg_norm,
    y_cls = y_cls,
    batch_size = 512
)

model = MTLArchitecture(
    input_dim= flux_clean.shape[1],
    latent_dim= 64,
    num_classes= 7
)

Train: 13161 | Val: 4386 | Test: 4387


running the experiment

In [28]:
tracking = train(
    model= model,
    train_loader= train_loader,
    val_loader= val_loader,
    n_epochs= 50,
    lr = 0.001,
    beta = 1.0
)

Training on cuda


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : continued_cookie_904
COMET INFO:     url                   : https://www.comet.com/alcatraz312/dissertation-ml/3b05bbc5cd414115ad3bb81457edc3b1
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     sigma/cls [50]    : (1.0103651285171509, 1.339369773864746)
COMET INFO:     sigma/recon [50]  : (0.5507772564888, 0.9895935654640198)
COMET INFO:     sigma/reg [50]    : (0.7119883894920349, 0.9896915555000305)
COMET INFO:     train/cls [50]    : (1.8284654092788697, 1.901782479286194)
COMET INFO:     train/kl [50]     : (1.0086150746246858e-06, 0.4795138642191887)
COMET INFO:     train/loss [50]   : (0.15896851062774658, 1.8912600755691529)
COMET INFO

#### Diagnosis

In [ ]:
print(flux.min(), flux.max(), np.isnan(flux).sum())

In [ ]:
# Check what you're actually passing in
print(f"flux mean: {flux.mean():.4f}")
print(f"flux std:  {flux.std():.4f}")

# Also check labels
print(f"y_reg NaN: {np.isnan(y_reg_norm).sum()}")
print(f"y_reg min: {y_reg_norm.min():.4f}  max: {y_reg_norm.max():.4f}")
print(f"y_cls unique: {np.unique(y_cls)}")

In [ ]:
min_max_list = []
for stars in tqdm(flux):
    min_max = []
    min_max.append(min(stars))
    min_max.append(max(stars))
    min_max_list.append(min_max)


Mean absolute deviation for outlier finding 

In [ ]:
median = np.median(flux, axis = 1, keepdims= True)
mad = np.median(np.abs(flux - median), axis = 1, keepdims= True)
std_mad = 1.4826 * mad

outlier_mask_mad = np.abs(flux - median) > 5 * std_mad

print(outlier_mask_mad.sum())
print(flux.shape[0] * flux.shape[1])
print(f"fraction of bad/outlier pixels : {outlier_mask_mad.sum()/(flux.shape[0] * flux.shape[1])}")

Clipping the flux vectors according to mean absolute deviation with 5 sigma boundaries

In [ ]:
flux_clipped = np.clip(flux, median - 5 * std_mad, median + 5 * std_mad)


In [ ]:
print(f"Minimum : {flux_clipped.min()}")
print(f"Maximum : {flux_clipped.max()}")

In [ ]:
print(f"Median range: {median.min():.2f} to {median.max():.2f}")
print(f"MAD range:    {mad.min():.4f} to {mad.max():.4f}")
print(f"Threshold upper: {(median + 5 * std_mad).max():.2f}")
print(f"Threshold lower: {(median - 5 * std_mad).min():.2f}")

checking for number of stars that failed the 5550 A normalization

In [ ]:
per_star_median = np.median(flux, axis= 1, keepdims= True)
print(f"Stars with median > 5 : {(per_star_median > 5).sum()}")
print(f"Stars with median > 10 : {(per_star_median > 10).sum()}")
print(f"Stars with median < 0 : {(per_star_median < 0).sum()}")

import matplotlib.pyplot as plt
plt.hist(per_star_median, bins=100)
plt.xlabel("Per-star median flux")
plt.ylabel("Count")
plt.title("Distribution of per-star median flux")
plt.show()

In [ ]:
per_star_median = np.median(flux , axis = 1)
good_mask = per_star_median < 5.0

flux_clean = flux[good_mask]
y_reg_clean = y_reg_norm[good_mask]
y_cls_clean = y_cls[good_mask]

print(good_mask.sum())
print(f"Dropped: {(~good_mask).sum()}")

In [ ]:
print(flux_clean.max())
print(flux_clean.min())

In [ ]:
big_flux_stars = flux_clean[(flux_clean > 10) | (flux_clean < -3.0)]
total_pixels = flux_clean.shape[1] * flux_clean.shape[0]

print(f"Fraction of bad pixels remaining : {len(big_flux_stars)/total_pixels}")

number of stars with these bad pixels 

In [ ]:
bad_pixel_mask = (flux_clean > 10) | (flux_clean < -3.0)
stars_with_bad_pixels = bad_pixel_mask.any(axis=1)
print(f"Stars with at least 1 bad pixel : {stars_with_bad_pixels.sum()}")
print(f"Stars with > 10 bad pixels      : {(bad_pixel_mask.sum(axis=1) > 10).sum()}")
print(f"Stars with > 100 bad pixels     : {(bad_pixel_mask.sum(axis=1) > 100).sum()}")
print(f"Stars with > 1000 bad pixels     : {(bad_pixel_mask.sum(axis=1) > 1000).sum()}")

Investigating the worst star


In [ ]:
bad_per_star = bad_pixel_mask.sum(axis=1)
print(f"Max bad pixels in one star  : {bad_per_star.max()}")
worst_star_idx = bad_per_star.argmax()
print(f"Worst star index: {worst_star_idx}, bad pixels: {bad_per_star[worst_star_idx]}")

In [ ]:
reference_grid = np.linspace(4000, 8500, 3800)
worst = flux_clean[worst_star_idx]

plt.figure(figsize=(12, 4))
plt.plot(reference_grid, worst, alpha=0.7)
plt.axhline(10,  color='r', linestyle='--', label='upper clip (+10)')
plt.axhline(-3, color='g', linestyle='--', label='lower clip (-3)')
plt.xlabel("Wavelength (Å)")
plt.ylabel("Normalized flux")
plt.title(f"Worst star — {bad_per_star[worst_star_idx]} bad pixels")
plt.legend()
plt.show()

Try clipping the wavelength and then see the star

In [ ]:
worst_clean = np.clip(worst, -3.0, 10.0)

In [ ]:
plt.plot(reference_grid, worst_clean, alpha= 0.7)

Checking the MK stars which have the bad pixels:

In [ ]:
flux.shape

In [ ]:
y_cls.shape

Identifying the spectral classes with bad pixels

In [ ]:
star_index_list = []
for i in range(len(flux)):

    star = flux[i]
    star = star[(star > 10) | (star < -3.0)]
    if len(star) > 0:
        star_index_list.append(i)


len(star_index_list)
class_list = [y_cls[i] for i in star_index_list]

In [ ]:
class_map = {"O": 0, "B": 1, "A": 2, "F": 3, "G": 4, "K": 5, "M": 6}
class_series = pd.Series(class_list)

In [ ]:
m_stars = class_series[class_series == 6].count()
k_stars = class_series[class_series == 5].count()
g_stars = class_series[class_series == 4].count()
f_stars = class_series[class_series == 3].count()
a_stars = class_series[class_series == 2].count()
b_stars = class_series[class_series == 1].count()
o_stars = class_series[class_series == 0].count()

print(f"M stars : {m_stars}")
print(f"K stars : {k_stars}")
print(f"G stars : {g_stars}")
print(f"F stars : {f_stars}")
print(f"A stars : {a_stars}")
print(f"B stars : {b_stars}")
print(f"O stars : {o_stars}")